In [15]:
# Instala as dependências no kernel atual do Jupyter
%pip install -q python-dotenv langchain-text-splitters langchain-openai langchain-chroma langchain-community langchain-core pypdf

Note: you may need to restart the kernel to use updated packages.


In [16]:
import os
from dotenv import load_dotenv

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [17]:
load_dotenv()

api_key = os.environ.get("OPENAI_API_KEY", "")
print("API Key carregada:", "✅" if api_key else "❌ Não encontrada no .env")

API Key carregada: ✅


In [18]:
embedding_model = OpenAIEmbeddings()
llm = ChatOpenAI(model="gpt-3.5-turbo", max_tokens=200)

In [19]:
pdf_link = "Practical-Machine-Learning-for-Computer-Vsion-End-to-End-Machine-Learning-for-Images.pdf"

loader = PyPDFLoader(pdf_link)
pages = loader.load_and_split()
print(f"Total de páginas carregadas: {len(pages)}")

Total de páginas carregadas: 479


In [20]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,
    chunk_overlap=20,
    length_function=len,
    add_start_index=True
)

chunks = text_splitter.split_documents(pages)
print(f"Total de chunks gerados: {len(chunks)}")

Total de chunks gerados: 479


In [21]:
vectorstore = Chroma.from_documents(documents=chunks, embedding=embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("Vector store criado com sucesso ✅")

Vector store criado com sucesso ✅


In [22]:
prompt = ChatPromptTemplate.from_template("""
Responda a pergunta abaixo com base apenas no contexto fornecido.
Se não souber a resposta, diga que não encontrou a informação no documento.

Contexto:
{context}

Pergunta: {question}
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [23]:
resposta = rag_chain.invoke("O que é Machine Learning para visão computacional?")
print(resposta)

Machine Learning para visão computacional é uma abordagem que utiliza métodos de aprendizado de máquina para extrair informações de imagens, imitando a capacidade cognitiva humana de reconhecer e classificar objetos em imagens.
